<a href="https://colab.research.google.com/github/gankur-oss/TestRerank/blob/main/Copy_of_RerankCrossEncoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers accelerate

In [ ]:
import torch
import os
import time

from sentence_transformers import CrossEncoder

In [ ]:
os.environ["OMP_NUM_THREADS"] = "8"
os.environ["MKL_NUM_THREADS"] = "8"

torch.set_num_threads(8)
torch.set_num_interop_threads(8)

In [ ]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [ ]:
model = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [ ]:
documents = [

f"Product {i}: cordless power drill with lithium battery designed for home repair DIY woodworking drilling tasks"
for i in range(10)

] + [

f"Product {i}: heavy duty hammer drill designed for concrete brick masonry drilling construction work"
for i in range(10,20)

] + [

f"Product {i}: waterproof ceramic bathroom wall tiles ideal for bathroom renovation and remodeling projects"
for i in range(20,30)

] + [

f"Product {i}: flexible garden watering hose pipe suitable for lawn irrigation and outdoor gardening"
for i in range(30,40)

] + [

f"Product {i}: cordless impact driver power tool for screw driving furniture assembly and carpentry"
for i in range(40,50)

] + [

f"Product {i}: LED ceiling panel lighting fixture energy efficient indoor home lighting solution"
for i in range(50,60)

] + [

f"Product {i}: electric circular saw power tool designed for wood cutting carpentry plywood boards"
for i in range(60,70)

] + [

f"Product {i}: paint roller kit with tray extension pole designed for wall painting and home decoration"
for i in range(70,80)

] + [

f"Product {i}: manual tile cutter tool designed for cutting ceramic and porcelain tiles precisely"
for i in range(80,90)

] + [

f"Product {i}: adjustable plumbing wrench tool designed for tightening metal pipes bolts and fittings"
for i in range(90,100)

]

print("Total documents:", len(documents))

Total documents: 100


In [ ]:
query = "best cordless drill for home DIY repair"

In [ ]:
pairs = [(query, doc[:256]) for doc in documents]

In [ ]:
_ = model.predict(pairs[:8], batch_size=8)

torch.cuda.synchronize()

In [ ]:
torch.cuda.synchronize()

start = time.time()

scores = model.predict(
    pairs,
    batch_size=128
)

torch.cuda.synchronize()

end = time.time()

latency_ms = (end - start) * 1000

print("Reranking latency:", round(latency_ms,2), "ms")

Reranking latency: 54.79 ms


In [ ]:
results = list(zip(documents, scores))

In [ ]:
reranked = sorted(
    results,
    key=lambda x: x[1],
    reverse=True
)

In [ ]:
for doc, score in reranked[:24]:

    print(round(score,3), "|", doc)

7.004 | Product 9: cordless power drill with lithium battery designed for home repair DIY woodworking drilling tasks
6.998 | Product 8: cordless power drill with lithium battery designed for home repair DIY woodworking drilling tasks
6.995 | Product 7: cordless power drill with lithium battery designed for home repair DIY woodworking drilling tasks
6.931 | Product 5: cordless power drill with lithium battery designed for home repair DIY woodworking drilling tasks
6.902 | Product 3: cordless power drill with lithium battery designed for home repair DIY woodworking drilling tasks
6.896 | Product 4: cordless power drill with lithium battery designed for home repair DIY woodworking drilling tasks
6.843 | Product 6: cordless power drill with lithium battery designed for home repair DIY woodworking drilling tasks
6.818 | Product 0: cordless power drill with lithium battery designed for home repair DIY woodworking drilling tasks
6.769 | Product 2: cordless power drill with lithium battery des